## Импорт Pandas

In [1]:
import pandas as pd

## Чтение лога

In [2]:
views = pd.read_csv(
    "../data/feed-views.log",
    sep="\t",
    names=["datetime","user"],
    engine="python",
    skiprows=[2,3],
    skipfooter=2
)


## Исходные типы

In [3]:
views.head()


Out[3]: 
                     datetime    user
0  2020-04-17 12:01:08.463179   artem
1  2020-04-17 12:01:23.743946   artem
2  2020-04-17 12:35:52.735016   artem
3  2020-04-17 12:36:21.401412  oksana
4  2020-04-17 12:36:22.023355  oksana


## Преобразование временных меток

In [4]:
views.dtypes

Out[4]: 
datetime    str
user        str
dtype: object


## Компоненты времени и категории суток

In [5]:
views["datetime"]=views["datetime"].astype("datetime64[ns]")
views.dtypes

Out[5]: 
datetime    datetime64[ns]
user                   str
dtype: object


## Количество наблюдений

In [6]:
bins=[0, 4, 7, 11, 17, 20, 24]
labels=(
    "night",
    "early morning",
    "morning",
    "afternoon",
    "early evening",
    "evening"
)
views["year"] = views["datetime"].dt.year
views["month"] = views["datetime"].dt.month
views["day"] = views["datetime"].dt.day
views["hour"] = views["datetime"].dt.hour
views["minute"] = views["datetime"].dt.minute
views["second"] = views["datetime"].dt.second
views["daytime"] = pd.cut(
    views["hour"],
    bins=bins,
    labels=labels,
    right=False
)
views = views.set_index("user")
views.head(10)

Out[6]: 
                            datetime  year  ...  second    daytime
user                                        ...                   
artem     2020-04-17 12:01:08.463179  2020  ...       8  afternoon
artem     2020-04-17 12:01:23.743946  2020  ...      23  afternoon
artem     2020-04-17 12:35:52.735016  2020  ...      52  afternoon
oksana    2020-04-17 12:36:21.401412  2020  ...      21  afternoon
oksana    2020-04-17 12:36:22.023355  2020  ...      22  afternoon
artem     2020-04-17 13:55:19.129243  2020  ...      19  afternoon
artem     2020-04-17 15:00:33.138530  2020  ...      33  afternoon
artem     2020-04-17 15:14:09.581054  2020  ...       9  afternoon
vladislav 2020-04-17 22:46:26.785035  2020  ...      26    evening
artem     2020-04-18 00:30:45.051569  2020  ...      45      night

[10 rows x 8 columns]


## Частоты категорий

In [7]:
views.count()

Out[7]: 
datetime    1072
year        1072
month       1072
day         1072
hour        1072
minute      1072
second      1072
daytime     1072
dtype: int64


## Сортировка по времени

In [8]:
views["daytime"].value_counts()

Out[8]: 
daytime
evening          508
afternoon        250
early evening    145
night            129
morning           35
early morning      5
Name: count, dtype: int64


## Максимальный час ночью

In [9]:
views = views.sort_values(
    ["hour", "minute", "second"],
)
views.head(100)

Out[9]: 
                            datetime  year  month  ...  minute  second daytime
user                                               ...                        
valentina 2020-05-15 00:00:13.222265  2020      5  ...       0      13   night
valentina 2020-05-15 00:01:05.153738  2020      5  ...       1       5   night
pavel     2020-05-12 00:01:27.764025  2020      5  ...       1      27   night
pavel     2020-05-12 00:01:38.444917  2020      5  ...       1      38   night
pavel     2020-05-12 00:01:55.395042  2020      5  ...       1      55   night
...                              ...   ...    ...  ...     ...     ...     ...
pavel     2020-05-10 01:26:52.205761  2020      5  ...      26      52   night
pavel     2020-05-10 01:28:55.202696  2020      5  ...      28      55   night
pavel     2020-05-10 01:29:00.562990  2020      5  ...      29       0   night
pavel     2020-05-10 01:29:03.914597  2020      5  ...      29       3   night
pavel     2020-05-10 01:29:08.880411  2020 

## Минимальный час утром

In [10]:
night_max_hour = views.loc[
    views["daytime"] == "night",
    "hour"
].max()
print(night_max_hour)

3


## Пользователи в выбранные часы

In [11]:
morning_min_hour = views.loc[
    views["daytime"] == "morning",
    "hour"
].min()
print(morning_min_hour)

8


## Ночной посетитель

In [12]:
night_users = views.loc[
    views["hour"]==night_max_hour
].index.unique()


## Утренний посетитель

In [13]:
morning_users = views.loc[
    views["hour"]==morning_min_hour
].index.unique()

## Мода

In [14]:
night_users[0]

Out[14]: 'konstantin'


## Самые ранние часы

In [15]:
morning_users[0]

Out[15]: 'alexander'


## Самые поздние часы

In [16]:
mode_hour=views["hour"].mode()[0]
mode_daytime=views["daytime"].mode()[0]


## Описательная статистика

In [17]:
print(mode_daytime)
print(mode_hour)

evening
22


## Межквартильный размах

In [18]:
views["hour"].nsmallest(3)

Out[18]: 
user
valentina    0
valentina    0
pavel        0
Name: hour, dtype: int32


## Шаг 19

In [19]:
views["hour"].nlargest(3)

Out[19]: 
user
ekaterina    23
ekaterina    23
ekaterina    23
Name: hour, dtype: int32


## Шаг 20

In [20]:
statistic = views.describe()
statistic

Out[20]: 
                            datetime    year  ...       minute       second
count                           1072  1072.0  ...  1072.000000  1072.000000
mean   2020-05-10 09:30:16.241518592  2020.0  ...    29.602612    29.508396
min       2020-04-17 12:01:08.463179  2020.0  ...     0.000000     0.000000
25%    2020-05-10 01:13:53.448644096  2020.0  ...    14.000000    14.000000
50%    2020-05-11 22:48:35.302552832  2020.0  ...    29.000000    30.000000
75%    2020-05-14 14:30:41.062071296  2020.0  ...    46.000000    45.000000
max       2020-05-21 23:23:49.995349  2020.0  ...    59.000000    59.000000
std                              NaN     0.0  ...    17.710471    17.424398

[8 rows x 7 columns]


## Шаг 21

In [21]:
iqr = statistic.loc["75%", "hour"] - statistic.loc["25%", "hour"]
iqr

Out[21]: np.float64(9.0)
